# Streamlit Interface Prototype

This notebook demonstrates a standalone Streamlit interface for the chatbot workflow.

The maintained application entry point is `main.py`, with the interface implemented in `app/ui.py`. The code in this notebook is retained as a development example; run the maintained application with `streamlit run main.py`.


### Shared project OpenAI client
Run `00_Project_Setup.ipynb` once first. This notebook then uses the shared project configuration; no API key is entered here.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'config').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from app.services.llm_service import get_client, MODEL_NAME

client = get_client()
print('Shared OpenAI client ready.')


Shared OpenAI client ready.


### Project-level LLM client
This notebook uses the API configuration initialized in `00_Project_Setup.ipynb`. No API key is entered here.

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'config').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from app.services.llm_service import get_client, MODEL_NAME

client = get_client()
print('Shared OpenAI client ready.')


Shared OpenAI client ready.


## 1. Install packages

Run this cell first.

In [3]:
%pip -q install -U streamlit openai requests pandas pyngrok

DEPRECATION: Loading egg at /Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/jupyter-1.0.0-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Create the Streamlit application

The application below is self-contained. It includes:

- secure API-key loading through an environment variable
- LLM request extraction
- location normalization
- Open-Meteo forecast retrieval
- activity-specific weather logic
- LLM response generation
- a simple chat interface

The application is saved as `app.py`.

In [4]:
%%writefile app.py
import os
import json
import requests
import pandas as pd
import streamlit as st
# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------


WEATHER_CODES = {
    0: "Clear sky",
    1: "Mainly clear",
    2: "Partly cloudy",
    3: "Overcast",
    45: "Fog",
    48: "Depositing rime fog",
    51: "Light drizzle",
    53: "Moderate drizzle",
    55: "Dense drizzle",
    61: "Slight rain",
    63: "Moderate rain",
    65: "Heavy rain",
    71: "Slight snow",
    73: "Moderate snow",
    75: "Heavy snow",
    80: "Slight rain showers",
    81: "Moderate rain showers",
    82: "Violent rain showers",
    85: "Slight snow showers",
    86: "Heavy snow showers",
    95: "Thunderstorm",
    96: "Thunderstorm with slight hail",
    99: "Thunderstorm with heavy hail"
}

ACTIVITY_PROFILES = {
    "hiking": {
        "max_rain_probability": 60,
        "max_wind_kmh": 35,
        "low_temp_c": 0,
        "high_temp_c": 30
    },
    "cycling": {
        "max_rain_probability": 50,
        "max_wind_kmh": 30,
        "low_temp_c": 2,
        "high_temp_c": 32
    },
    "mountain biking": {
        "max_rain_probability": 45,
        "max_wind_kmh": 30,
        "low_temp_c": 0,
        "high_temp_c": 28
    },
    "running": {
        "max_rain_probability": 50,
        "max_wind_kmh": 30,
        "low_temp_c": 0,
        "high_temp_c": 30
    },
    "trail running": {
        "max_rain_probability": 45,
        "max_wind_kmh": 30,
        "low_temp_c": 0,
        "high_temp_c": 28
    },
    "camping": {
        "max_rain_probability": 50,
        "max_wind_kmh": 25,
        "low_temp_c": 5,
        "high_temp_c": 30
    },
    "outdoor tour": {
        "max_rain_probability": 50,
        "max_wind_kmh": 30,
        "low_temp_c": 2,
        "high_temp_c": 32
    }
}

REQUEST_SCHEMA = {
    "type": "object",
    "properties": {
        "activity": {"type": "string"},
        "location": {"type": "string"},
        "date": {"type": "string"},
        "date_iso": {"type": "string"},
        "time_period": {"type": "string"},
        "elevation_m": {"type": ["number", "null"]}
    },
    "required": [
        "activity",
        "location",
        "date",
        "date_iso",
        "time_period",
        "elevation_m"
    ],
    "additionalProperties": False
}


# ---------------------------------------------------------
# Helper functions
# ---------------------------------------------------------

def get_client():
    api_key = os.getenv("OPENAI_API_KEY")

    if not api_key:
        raise ValueError(
            "OPENAI_API_KEY is not available. "
            "Set it as an environment variable before starting Streamlit."
        )

    return OpenAI(api_key=api_key)


def normalize_location(location):
    cleaned = str(location).strip()

    prefixes = [
        "near ",
        "around ",
        "in ",
        "at ",
        "the "
    ]

    changed = True

    while changed:
        changed = False
        lower = cleaned.lower()

        for prefix in prefixes:
            if lower.startswith(prefix):
                cleaned = cleaned[len(prefix):].strip()
                changed = True
                break

    return cleaned


def geocode_location(location):
    url = "https://geocoding-api.open-meteo.com/v1/search"

    params = {
        "name": normalize_location(location),
        "count": 1,
        "language": "en",
        "format": "json"
    }

    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()

    data = response.json()

    if "results" not in data or not data["results"]:
        raise ValueError(
            f"Location could not be found: {location}"
        )

    result = data["results"][0]

    return {
        "name": result["name"],
        "latitude": result["latitude"],
        "longitude": result["longitude"],
        "country": result.get("country"),
        "timezone": result.get("timezone", "auto")
    }


def get_weather_forecast(
    latitude,
    longitude,
    timezone="auto",
    forecast_days=7
):
    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": ",".join([
            "temperature_2m",
            "precipitation_probability",
            "precipitation",
            "weather_code",
            "cloud_cover",
            "wind_speed_10m"
        ]),
        "timezone": timezone,
        "forecast_days": forecast_days,
        "temperature_unit": "celsius",
        "wind_speed_unit": "kmh",
        "precipitation_unit": "mm"
    }

    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()

    return response.json()


def weather_code_to_description(code):
    return WEATHER_CODES.get(
        int(code),
        "Unknown weather condition"
    )


def assess_activity_weather(
    activity,
    temperature,
    wind_speed,
    precipitation_probability,
    weather_description=None
):
    activity_key = activity.lower().strip()

    if activity_key not in ACTIVITY_PROFILES:
        # Generic outdoor-tour profile for an unlisted activity.
        activity_key = "outdoor tour"

    profile = ACTIVITY_PROFILES[activity_key]
    concerns = []

    if temperature < profile["low_temp_c"]:
        concerns.append("low temperature")

    if temperature > profile["high_temp_c"]:
        concerns.append("high temperature")

    if wind_speed > profile["max_wind_kmh"]:
        concerns.append("strong wind")

    if precipitation_probability > profile["max_rain_probability"]:
        concerns.append("high precipitation probability")

    return {
        "activity": activity_key,
        "temperature_c": round(temperature, 1),
        "wind_kmh": round(wind_speed, 1),
        "precipitation_probability": round(
            precipitation_probability, 1
        ),
        "weather_description": weather_description,
        "concerns": concerns,
        "summary": (
            "Main factors: " + ", ".join(concerns)
            if concerns
            else "No major weather concern identified by the prototype rules."
        )
    }


# ---------------------------------------------------------
# LLM request extraction
# ---------------------------------------------------------

def extract_weather_request(client, user_question):
    from datetime import date

    today = date.today()

    instructions = f"""
You are the request-understanding component of a weather forecast chatbot.

Today's date is {today.isoformat()}.

Extract:
- activity
- location
- date as stated by the user
- date_iso as YYYY-MM-DD
- time period
- elevation in metres when explicitly provided

Supported activities:
hiking, cycling, mountain biking, running, camping,
trail running, outdoor tour.

Rules:
1. Extract only information supported by the user's wording.
2. Do not invent missing values.
3. Resolve relative dates using today's date.
4. For Saturday or Sunday, return the next occurrence.
5. For this weekend, use the upcoming Saturday.
6. If no time period is given, return an empty string.
7. If no elevation is given, return null.
8. Return only the structured data.
"""

    response = client.responses.create(
        model=MODEL_NAME,
        instructions=instructions,
        input=user_question,
        text={
            "format": {
                "type": "json_schema",
                "name": "weather_request",
                "schema": REQUEST_SCHEMA,
                "strict": True
            }
        }
    )

    return json.loads(response.output_text)


# ---------------------------------------------------------
# Forecast selection
# ---------------------------------------------------------

def select_forecast_period(
    forecast_df,
    date_iso,
    time_period
):
    df = forecast_df.copy()
    df["datetime"] = pd.to_datetime(df["time"])

    target_date = pd.to_datetime(
        date_iso
    ).date()

    df = df[
        df["datetime"].dt.date == target_date
    ]

    period = time_period.lower().strip()

    if period == "morning":
        df = df[
            df["datetime"].dt.hour.between(6, 11)
        ]

    elif period == "afternoon":
        df = df[
            df["datetime"].dt.hour.between(12, 17)
        ]

    elif period == "evening":
        df = df[
            df["datetime"].dt.hour.between(18, 21)
        ]

    elif period == "overnight":
        df = df[
            (df["datetime"].dt.hour >= 22)
            | (df["datetime"].dt.hour <= 5)
        ]

    return df


def summarize_forecast(selected_df):
    if selected_df.empty:
        return None

    return {
        "temperature_min_c": round(
            float(selected_df["temperature_2m"].min()), 1
        ),
        "temperature_max_c": round(
            float(selected_df["temperature_2m"].max()), 1
        ),
        "wind_max_kmh": round(
            float(selected_df["wind_speed_10m"].max()), 1
        ),
        "precipitation_probability_max": round(
            float(
                selected_df[
                    "precipitation_probability"
                ].max()
            ), 1
        ),
        "precipitation_total_mm": round(
            float(
                selected_df["precipitation"].sum()
            ), 1
        ),
        "conditions": list(
            dict.fromkeys(
                weather_code_to_description(code)
                for code in selected_df[
                    "weather_code"
                ].astype(int)
            )
        )
    }


# ---------------------------------------------------------
# Final LLM response
# ---------------------------------------------------------

def generate_weather_response(
    client,
    user_question,
    request,
    location,
    forecast_summary,
    activity_assessment
):
    prompt = f"""
User question:
{user_question}

Structured request:
{json.dumps(request, ensure_ascii=False)}

Location:
{json.dumps(location, ensure_ascii=False)}

Forecast:
{json.dumps(forecast_summary, ensure_ascii=False)}

Activity assessment:
{json.dumps(activity_assessment, ensure_ascii=False)}

Write a concise answer for the outdoor user.

Requirements:
- Use only supplied forecast data for factual weather values.
- Mention temperature, conditions and wind.
- Mention precipitation when relevant.
- Address the requested activity.
- Do not claim guaranteed safety.
- Do not invent missing information.
"""

    response = client.responses.create(
        model=MODEL_NAME,
        instructions=(
            "You generate concise, factual weather answers. "
            "Ground factual weather statements in the supplied data."
        ),
        input=prompt
    )

    return response.output_text


# ---------------------------------------------------------
# Complete chatbot pipeline
# ---------------------------------------------------------

def run_chatbot(user_question):
    client = get_client()

    request = extract_weather_request(
        client,
        user_question
    )

    original_location = request["location"]

    request["location"] = normalize_location(
        request["location"]
    )

    location = geocode_location(
        request["location"]
    )

    weather_data = get_weather_forecast(
        latitude=location["latitude"],
        longitude=location["longitude"],
        timezone=location["timezone"],
        forecast_days=7
    )

    forecast_df = pd.DataFrame(
        weather_data["hourly"]
    )

    selected_df = select_forecast_period(
        forecast_df,
        request["date_iso"],
        request["time_period"]
    )

    forecast_summary = summarize_forecast(
        selected_df
    )

    if forecast_summary is None:
        raise ValueError(
            "No forecast data is available for "
            f"{request['date_iso']}."
        )

    average_temperature = (
        forecast_summary["temperature_min_c"]
        + forecast_summary["temperature_max_c"]
    ) / 2

    assessment = assess_activity_weather(
        request["activity"],
        average_temperature,
        forecast_summary["wind_max_kmh"],
        forecast_summary[
            "precipitation_probability_max"
        ],
        ", ".join(
            forecast_summary["conditions"]
        )
    )

    answer = generate_weather_response(
        client,
        user_question,
        request,
        location,
        forecast_summary,
        assessment
    )

    return {
        "answer": answer,
        "request": request,
        "original_location": original_location,
        "location": location,
        "forecast": forecast_summary,
        "assessment": assessment
    }


# ---------------------------------------------------------
# Streamlit interface
# ---------------------------------------------------------

st.set_page_config(
    page_title="Weather Forecast Chatbot",
    page_icon="🌦️",
    layout="centered"
)

st.title("🌦️ Weather Forecast Chatbot")
st.caption(
    "AI-powered weather information for outdoor activities"
)

st.write(
    "Ask about hiking, cycling, running, camping, "
    "mountain biking, or another outdoor activity."
)

if "messages" not in st.session_state:
    st.session_state.messages = []

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

user_question = st.chat_input(
    "Example: Will it rain during my cycling tour in Berlin tomorrow afternoon?"
)

if user_question:
    st.session_state.messages.append({
        "role": "user",
        "content": user_question
    })

    with st.chat_message("user"):
        st.markdown(user_question)

    with st.chat_message("assistant"):
        with st.spinner("Checking the weather..."):
            try:
                result = run_chatbot(
                    user_question
                )

                st.markdown(
                    result["answer"]
                )

                with st.expander(
                    "Technical details"
                ):
                    st.write(
                        "Structured request:"
                    )
                    st.json(
                        result["request"]
                    )

                    st.write(
                        "Forecast:"
                    )
                    st.json(
                        result["forecast"]
                    )

                    st.write(
                        "Activity assessment:"
                    )
                    st.json(
                        result["assessment"]
                    )

            except Exception as error:
                st.error(
                    "The request could not be processed."
                )

                with st.expander(
                    "Technical error"
                ):
                    st.exception(error)

                result = None

    if result:
        st.session_state.messages.append({
            "role": "assistant",
            "content": result["answer"]
        })

st.divider()

st.caption(
    "Forecast data is retrieved "
    "from Open-Meteo. Activity rules are prototype heuristics "
    "and are not safety guarantees."
)


Writing app.py


## 3. Set the OpenAI API key for the Streamlit process

Colab Secrets are not automatically exposed to a separate Streamlit process. This cell retrieves the secret and places it into the current process environment.

The key is **not printed**.

## 4. Quick syntax check

Before launching Streamlit, make sure `app.py` can be compiled.

In [5]:
import py_compile
py_compile.compile("app.py", doraise=True)
print("app.py syntax check: OK")

app.py syntax check: OK


## 5. Start Streamlit

This launches the local Streamlit server in the Colab runtime.

In [6]:
import subprocess
import time

streamlit_process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "app.py",
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

print("Streamlit process started.")
print("PID:", streamlit_process.pid)

Streamlit process started.
PID: 3011


## 6. Create a temporary public tunnel

For a Colab demonstration, we use `pyngrok` to expose port 8501.

If your environment asks for an ngrok authentication token, create/configure it in ngrok and then rerun this cell.

**Do not paste an OpenAI API key here.**

In [7]:
from pyngrok import ngrok

try:
    public_url = ngrok.connect(8501)
    print("Streamlit URL:")
    print(public_url)
except Exception as e:
    print("Could not create the tunnel.")
    print("You can still run the Streamlit app locally with:")
    print("streamlit run app.py")
    print()
    print("Technical message:", e)

Could not create the tunnel.
You can still run the Streamlit app locally with:
streamlit run app.py

Technical message: An error occurred while downloading ngrok from https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-darwin-arm64.zip: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:992)>


# 7. Test the chatbot

Open the Streamlit URL printed above.

Start with:

> **Will it rain during my cycling tour in Berlin tomorrow afternoon?**

Then try:

> **What's the weather like for my hike in Munich tomorrow?**

The **Technical details** expander lets you demonstrate the structured request, forecast summary, and activity assessment behind the natural-language response.

# 8. What this demonstrates for application

The Streamlit prototype demonstrates the application architecture as an actual user-facing application:

- natural-language user input
- LLM-based request understanding
- external weather API
- activity-specific processing
- generated natural-language response
- transparent technical details for the prototype

### Important POC limitation

The Streamlit application is currently intended for demonstration in the Colab environment. A later packaging/deployment step can move the same `app.py` to a normal Python environment.

# 9. Next development step

After the interface works, the next work should focus on **test evidence and project documentation**:

1. capture screenshots of the working chatbot;
2. document the 8/10 initial test and the location-normalization improvement;
3. rerun the 10 scenarios after the fix;
4. record the final test results;
5. prepare the approximately 10-slide project documentation required by the application.

The underlying architecture remains the one defined in initial design.